# YAD-AfroXLMR — Evaluation

Reproduces the accuracy and output-preservation results of the paper from the released prediction files. It needs no GPU and no model weights, and runs in under a minute.

1. Load the input, the reference and the seven prediction files
2. Results table (CER, WER, DER, WDER, BLEU, ChrF)
3. Data versus architecture
4. Base-letter preservation against the input
5. Combining marks in each file
6. Words mT5 changed

The scorer lives in `yad_eval.py`, which `evaluate.py` also uses, so the notebook and the script cannot drift apart. Result tables are written to `results/`.

**Run locally:** `pip install -r requirements.txt`, then open this notebook from the repository root.
**Run on Colab:** run the first code cell, which clones the repository.

## 1. Setup

In [1]:
import os, sys

# On Colab, get the repository first. Locally this cell does nothing.
#   - If yad-afroxlmr-eval*.zip has been uploaded anywhere under /content, it is unzipped and used.
#   - Otherwise the repository is cloned from GitHub.
REPO_URL = 'https://github.com/JehohshuaA/yad-afroxlmr-eval.git'
if 'google.colab' in sys.modules and not os.path.exists('yad_eval.py'):
    import glob
    found = glob.glob('/content/**/yad_eval.py', recursive=True)
    zips = sorted(glob.glob('/content/**/yad-afroxlmr-eval*.zip', recursive=True))
    if not found and zips:
        !unzip -q -o "{zips[-1]}" -d /content
        found = glob.glob('/content/**/yad_eval.py', recursive=True)
    if not found:
        !git clone -q {REPO_URL} /content/yad-afroxlmr-eval
        found = glob.glob('/content/**/yad_eval.py', recursive=True)
    if not found:
        raise RuntimeError(f'Could not get the repository. Check that {REPO_URL} exists and is public, '
                           'or upload yad-afroxlmr-eval.zip (folder icon, left), wait for the upload '
                           'to finish, and run this cell again.')
    %cd {os.path.dirname(found[0])}
    !pip install -q -r requirements.txt

import unicodedata
from yad_eval import (SYSTEMS, LABELS, MARKS, read_lines, save_csv, score_folded,
                      preservation, mark_counts, changed_spans)

OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

src  = [l.strip() for l in read_lines('data/unyo.txt')]   # undiacritized input
gold = read_lines('data/dcyo.txt')                         # diacritized reference
preds = {s: read_lines(f'predictions/{s}.txt') for s in SYSTEMS}

assert len(src) == len(gold) == 3330
for s, p in preds.items():
    assert len(p) == 3330, f'{s}: {len(p)} lines'
print(f'Loaded input, reference and {len(preds)} prediction files, 3,330 lines each.')

Loaded input, reference and 7 prediction files, 3,330 lines each.


## 2. Results table

U+0329 (combining vertical line below) is folded into U+0323 (combining dot below) in the reference and every prediction before scoring, because both write the same Yorùbá underdot. All metrics are computed over the whole corpus, not averaged per sentence. Long sentences were labelled in windows of at most 254 tokens (see `efficiency.ipynb`), so no part of any sentence was left unlabelled.

A wrong letter makes its word wrong, so CER should come out below WER. The check at the end flags any row where it does not.

In [2]:
final, rows = {}, []
print(f"{'System':<22}{'CER':>8}{'WER':>8}{'DER':>8}{'WDER':>8}{'BLEU':>8}{'ChrF':>8}")
print('-' * 70)
for s in SYSTEMS:
    r = score_folded(gold, preds[s])
    final[s] = r
    rows.append({'system': s, **{k: round(v, 4) for k, v in r.items()}})
    print(f'{LABELS[s]:<22}' + ''.join(f'{r[k]:>8.4f}' for k in r))

bad = [s for s, r in final.items() if r['CER'] >= r['WER']]
print('\nCER < WER on every row.' if not bad else f'\nCER >= WER for: {bad}')
save_csv(rows, f'{OUT_DIR}/final_results.csv');

System                     CER     WER     DER    WDER    BLEU    ChrF
----------------------------------------------------------------------


B0  MENYO-20k only      0.0617  0.2260  0.1400  0.2305  0.5502  0.7718


B1  + secular           0.0479  0.1808  0.1088  0.1851  0.6284  0.8172


B2  + Mormon/Qur'an     0.0445  0.1696  0.1014  0.1742  0.6473  0.8279


B3  + Bible (full)      0.0424  0.1583  0.0997  0.1647  0.6678  0.8338


A   full + Igbo         0.0410  0.1537  0.0956  0.1591  0.6760  0.8397


C   no Phase 1          0.0418  0.1565  0.0973  0.1622  0.6717  0.8365


mT5 (greedy)            0.0462  0.1423  0.0796  0.1371  0.7042  0.8542

CER < WER on every row.


## 3. Data versus architecture

- **Data gain:** DER improvement from the smallest training corpus (B0) to the full corpus (B3), with the architecture fixed.
- **Architecture spread:** the range of DER across the three configurations trained on the full corpus (A, B3, C), with the data fixed.

In [3]:
gain = final['B0']['DER'] - final['B3']['DER']
arch = {k: final[k]['DER'] for k in ('A', 'B3', 'C')}
spread = max(arch.values()) - min(arch.values())
best = min(arch, key=arch.get)

for k in ('B0', 'B1', 'B2', 'B3'):
    print(f'   {LABELS[k]:<22} DER {100*final[k]["DER"]:.2f}%')
print()
for k, v in arch.items():
    print(f'   {LABELS[k]:<22} DER {100*v:.2f}%')

print(f'\nData gain (B0 -> B3):          {100*gain:.2f} points')
print(f'Architecture spread (A/B3/C):  {100*spread:.2f} points')
print(f'Ratio:                         {gain/spread:.1f}x')
print(f'Best configuration ({best}) vs mT5: DER {100*arch[best]:.2f}% vs {100*final["mT5"]["DER"]:.2f}%')

save_csv([{'data gain (pts)': round(100*gain, 2), 'architecture spread (pts)': round(100*spread, 2),
           'ratio': round(gain/spread, 1), 'best config': best,
           'best config DER (%)': round(100*arch[best], 2),
           'mT5 DER (%)': round(100*final['mT5']['DER'], 2)}], f'{OUT_DIR}/data_vs_architecture.csv');

   B0  MENYO-20k only     DER 14.00%
   B1  + secular          DER 10.88%
   B2  + Mormon/Qur'an    DER 10.14%
   B3  + Bible (full)     DER 9.97%

   A   full + Igbo        DER 9.56%
   B3  + Bible (full)     DER 9.97%
   C   no Phase 1         DER 9.73%

Data gain (B0 -> B3):          4.03 points
Architecture spread (A/B3/C):  0.41 points
Ratio:                         9.7x
Best configuration (A) vs mT5: DER 9.56% vs 7.96%


## 4. Base-letter preservation

Each output is compared with the undiacritized **input**, not the reference.

| Column | Meaning |
|---|---|
| lines unchanged | output with every mark removed is identical to the input |
| lines with words changed | the words (letters and digits only, lower-cased) differ from the input's |
| words altered | input words replaced or deleted, found by aligning the two word sequences |
| non-Yorùbá marks | combining marks outside Yorùbá orthography |

The token-classification configurations can only attach marks to characters already in the input, so they should show 3,330 unchanged lines.

In [4]:
pres = []
print(f"{'System':<6}{'unchanged':>11}{'words changed':>15}{'words altered':>22}{'non-Yoruba':>12}")
for s in SYSTEMS:
    r = preservation(src, preds[s])
    pres.append({'system': s, **r})
    print(f"{s:<6}{r['lines unchanged']:>11,}{r['lines with words changed']:>15,}"
          f"{r['words altered']:>12,} ({r['words altered %']:.2f}%){r['non-Yoruba marks']:>12}")
print(f"(lines: 3,330; input words: {pres[0]['words total']:,})")
save_csv(pres, f'{OUT_DIR}/preservation.csv');

System  unchanged  words changed         words altered  non-Yoruba


B0          3,330              0           0 (0.00%)           0


B1          3,330              0           0 (0.00%)           0


B2          3,330              0           0 (0.00%)           0


B3          3,330              0           0 (0.00%)           0


A           3,330              0           0 (0.00%)           0
C           3,330              0           0 (0.00%)           0


mT5         1,954            826       1,311 (1.72%)          21
(lines: 3,330; input words: 76,122)


## 5. Combining marks in each file

Yorùbá uses the grave (U+0300), the acute (U+0301) and the underdot, which may be written as U+0323 or U+0329. The reference itself contains three marks outside the orthography (two U+0308, one U+0328). They are in the published YAD test set and are left unchanged.

In [5]:
mrows = []
print(f"{'File':<6}{'grave':>8}{'acute':>8}{'U+0323':>8}{'U+0329':>8}{'other':>7}  detail")
for name, lines in [('gold', gold)] + [(s, preds[s]) for s in SYSTEMS]:
    c = mark_counts(lines)
    foreign = {ch: n for ch, n in c.items() if ch not in MARKS}
    detail = ', '.join(f"U+{ord(ch):04X} {unicodedata.name(ch, '?').title()} x{n}" for ch, n in foreign.items())
    row = {'file': name, 'U+0300': c.get('\u0300', 0), 'U+0301': c.get('\u0301', 0),
           'U+0323': c.get('\u0323', 0), 'U+0329': c.get('\u0329', 0),
           'non-Yoruba marks': sum(foreign.values()), 'non-Yoruba detail': detail}
    mrows.append(row)
    print(f"{name:<6}{row['U+0300']:>8}{row['U+0301']:>8}{row['U+0323']:>8}{row['U+0329']:>8}"
          f"{row['non-Yoruba marks']:>7}  {detail}")
save_csv(mrows, f'{OUT_DIR}/mark_counts.csv');

File     grave   acute  U+0323  U+0329  other  detail
gold     49918   55009   32673    1195      3  U+0308 Combining Diaeresis x2, U+0328 Combining Ogonek x1
B0       50015   55913   34563       0      0  
B1       49967   55078   34175       0      0  
B2       49842   55456   34017       0      0  
B3       49284   55036   33766       0      0  


A        49271   55273   33797       0      0  


C        49370   55184   33745       0      0  
mT5      50226   54987   33843     135     21  U+0302 Combining Circumflex Accent x6, U+0303 Combining Tilde x4, U+0327 Combining Cedilla x3, U+0304 Combining Macron x3, U+0308 Combining Diaeresis x2, U+0306 Combining Breve x2, U+030C Combining Caron x1


## 6. Words mT5 changed

Spans where mT5's output differs from the reference in its letters (case and punctuation ignored), with Configuration A's output for the same span. Our model never changes letters, so its column shows the input word with predicted diacritics. Insertions have an empty reference span, so the count of reference spans can be lower than the count of changes.

In [6]:
spans = changed_spans(gold, preds['mT5'], preds['A'])
ref_spans = [r['gold'] for r in spans if r['gold']]
caps = sum(w[:1].isupper() for w in ref_spans)
print(f'Changed spans: {len(spans):,}')
print(f'Reference spans starting with a capital (mostly names): {caps:,} / {len(ref_spans):,} '
      f'({100*caps/len(ref_spans):.1f}%)\n')

print(f"{'line':>5}  {'reference':<28}{'mT5':<28}{'ours (A)':<28}")
for r in spans[:25]:
    print(f"{r['line']:>5}  {r['gold'][:26]:<28}{r['mT5'][:26]:<28}{r['ours (A)'][:26]:<28}")
save_csv(spans, f'{OUT_DIR}/mt5_changed_words.csv');

Changed spans: 1,001
Reference spans starting with a capital (mostly names): 483 / 997 (48.4%)

 line  reference                   mT5                         ours (A)                    
    0  Ìbàdàn                      Ibojàn                      Ìbàdàn                      
    2  Cítì                        Bitì                        Citi                        
    3  Moi.                        Moro.                       Moi.                        
    5  Bauchi.                     Bábélì.                     Bauchi.                     
    6  ìkẹtàdíláàádọ́ta            ìkẹtàléláàádọ́ta            ìkẹtadíláàádọ́ta            
    7  NARTO                       Nasset                      NARTO                       
   10  Sherman                     S Harbor                    Sherman                     
   22  gbíngbin-ohun-ọ̀gbìn-nínú-  gbígbin-ohun-ọ̀gbìn-nínú-o  gbingbin-ohun-ọ̀gbìn-nínú-  
   25  gbọ̀gàn                     gbọ̀ngàn                    gbógan       